In [1]:
from all_needed_modules import *
from all_FFOPT_classes import *

system tools imported
basic tools imported
chemistry tools imported
ML  imported
all modules and data extracted


In [2]:
#THIS ARE ALL INFORMATIONS THAT WILL BE MOVED AND SHOULD BELONG IN A STRUCTURE_INFORMATION.PY FILE
Structure_name = 'GF334'
structure_carfile = './data/GF334_v2/GF334.car'
structure_mdffile = './data/GF334_v2/GF334.mdf'
DFT_ref_path = './data/GF334_v2/DFT_reference'
GMX_ref_path = '/home/uccaset/Scratch/FFOPTI/data/GF334_v2/classical'
charge_file = '/home/uccaset/Scratch/FFOPTI/data/GF334_v2/REPEAT_charges.resp'
groupid='MOF'
atom_types = ['Zn3', 'C_2', 'C_1', 'H_', 'N_R']  # fixed order is important
size = [26.494, 26.494, 29.203]

In [3]:
#information extraction
Structure = About_Structure(name=Structure_name, carfile=structure_carfile, mdffile=structure_mdffile, chargefile=charge_file)
DFT_ref = DFT_reference(Structure, Structure_name, DFT_ref_path, structure_file='GF334.pdb')
GMX_ref = GMX_read_reference(Structure_name, GMX_ref_path)
initial_opti_info = ParOpti_preparation(Structure_name,Structure, DFT_ref_path, GMX_ref_path)
print('information obtained')

information obtained


In [4]:
#Obtaining the initial GMX parameters
param = GMX_ref.ref_FF_parameters()
print('param extracted')
print(param)

Obtained force field parameters for 2 unique parameter sets.
param extracted
{'D1': {'lj':   atom func    mass  charge ptype     sigma  epsilon
0  Zn3    1  65.380     0.0     A  0.246155   0.5188
1  C_2    1  12.011     0.0     A  0.343085   0.4393
2  C_1    1  12.011     0.0     A  0.343085   0.4393
3   H_    1   1.008     0.0     A  0.257113   0.1841
4  N_R    1  14.007     0.0     A  0.326069   0.2887, 'bonds':   atom1 atom2 func  length  force_const
0   Zn3   N_R    1  0.1883    491035.36
1   C_2   C_2    1  0.1444    343088.00
2   C_2   C_1    1  0.1490    334720.00
3   C_1   N_R    1  0.1335    410032.00
4   C_1    H_    1  0.1080    284512.00
5   N_R   N_R    1  0.1320    418400.00, 'angles':   atom1 atom2 func atom3   angle  force_const
0   Zn3   N_R  C_1     1  132.60     1615.840
1   N_R   Zn3  N_R     1  109.47      301.050
2   Zn3   N_R  N_R     1  121.90     1615.840
3   C_1   C_2  C_2     1  112.70      488.273
4   C_2   C_1   H_     1  130.70      292.880
5   C_2   C_1 

In [5]:
traj_dcd_path = f'{DFT_ref_path}/md-pos-1.dcd'
frame_list = read(traj_dcd_path, index=':') #turning each frames into ase objects
random.seed(9191)
idx = random.sample(range(len(frame_list)), 100)
frames = [frame_list[i] for i in idx]
ref = frames[0] #choosing a reference frame for NVT to set PBC and cell dimensions

size = [26.494, 26.494, 29.203]

for atoms in frames:
    atoms.set_cell(size)
    atoms.set_pbc(True)
    atoms.wrap()  # optional, wraps positions inside box

In [7]:
# KEEPING TABS OF INDEX OF FRAMES SELECTED TO ENSURE CONSISTENCY
os.makedirs(f"{GMX_ref_path}/ffopti_backup", exist_ok=True)
with open(f"{GMX_ref_path}/ffopti_backup/numbers.json", "w") as f:
    json.dump(idx, f)

In [6]:
# Read from file
with open(f"{GMX_ref_path}/ffopti_backup/numbers.json", "r") as f:
    indx = json.load(f)

In [7]:
for atoms in frames:
    atoms.set_cell(size)
    atoms.set_pbc(True)
    atoms.wrap()  # optional, wraps positions inside box

In [8]:
tools = FFOPTI_tools()

## Creating GROMACS input files

In [12]:
for a, l1 in enumerate(frames):    
    frame_number = f'F{idx[a]}'
    for b, l2 in enumerate(param):
        D_number = f'D{b+1}'
            # Create the directory D*/F*D* if it doesn't exist
        folder_path = os.path.join(GMX_ref_path, D_number, f'{frame_number}{D_number}')
        os.makedirs(folder_path, exist_ok=True)
        FFOPTI_tools.write_gro(Structure_name, l1, f'{folder_path}', Structure)
        param_list = param[f'{D_number}']
            #force_field.itp 
        FFOPTI_tools.ref_ffitp_remake(f'{folder_path}', param_list)
            #MOF.itp
        subprocess.run(["cp", f"{GMX_ref_path}/MOF.itp", f"{folder_path}"],check=True)
        #initial_opti_info.write_gmx_structure_itp_simple(print_directory=f'{folder_path}')
            #topol.itp
        Structure.write_top_file(print_directory=f'{folder_path}')
            #SP.mdp 
        initial_opti_info.write_mdp_file(print_directory=f'{folder_path}')
print('initial files created')

grofile created
force_field.itp file printed
topol.top file printed
/home/uccaset/Scratch/FFOPTI/data/GF334_v2/classical/D1/F1157D1/SP.mdp written
grofile created
force_field.itp file printed
topol.top file printed
/home/uccaset/Scratch/FFOPTI/data/GF334_v2/classical/D2/F1157D2/SP.mdp written
grofile created
force_field.itp file printed
topol.top file printed
/home/uccaset/Scratch/FFOPTI/data/GF334_v2/classical/D1/F2869D1/SP.mdp written
grofile created
force_field.itp file printed
topol.top file printed
/home/uccaset/Scratch/FFOPTI/data/GF334_v2/classical/D2/F2869D2/SP.mdp written
grofile created
force_field.itp file printed
topol.top file printed
/home/uccaset/Scratch/FFOPTI/data/GF334_v2/classical/D1/F1158D1/SP.mdp written
grofile created
force_field.itp file printed
topol.top file printed
/home/uccaset/Scratch/FFOPTI/data/GF334_v2/classical/D2/F1158D2/SP.mdp written
grofile created
force_field.itp file printed
topol.top file printed
/home/uccaset/Scratch/FFOPTI/data/GF334_v2/classic

## Gromacs Calculations

In [9]:
#Submitting all the jobs
for a, l1 in enumerate(param):
    D_val = f'D{a+1}'
    tools.job_script_mpi(Structure_name, GMX_ref_path, D_val, indx)
    subprocess.run(
    ["bash", "-l", f'{GMX_ref_path}/{D_val}/run_all'],
    check=True)

/home/uccaset/Scratch/FFOPTI/data/GF334_v2/classical/D1/run_all written


Unloading compilers/gnu/10.2.0
  Unloading dependent: cp2k/8.2/ompi/gnu-10.2.0 mpi/openmpi/4.0.5/gnu-10.2.0
Unloading compilers/intel/2018/update3
  Unloading dependent: gromacs/2019.3/plumed/intel-2018 plumed/2.5.2/intel-2018

Unloading gcc-libs/10.2.0
  Unloading dependent: flex/2.5.39 libmatheval/1.1.11
    openblas/0.3.13-openmp/gnu-10.2.0 ucx/1.9.0/gnu-10.2.0
    binutils/2.36.1/gnu-10.2.0
Unloading compilers/gnu/10.2.0
  Unloading dependent: cp2k/8.2/ompi/gnu-10.2.0 mpi/openmpi/4.0.5/gnu-10.2.0
                      :-) GROMACS - gmx grompp, 2019.3 (-:

                            GROMACS is written by:
     Emile Apol      Rossen Apostolov      Paul Bauer     Herman J.C. Berendsen
    Par Bjelkmar      Christian Blau   Viacheslav Bolnykh     Kevin Boyd    
 Aldert van Buuren   Rudi van Drunen     Anton Feenstra       Alan Gray     
  Gerrit Groenhof     Anca Hamuraru    Vincent Hindriksen  M. Eric Irrgang  
  Aleksei Iupinov   Christoph Junghans     Joe Jordan     Dimitrios Kark

Analysing residue names:
There are:     1      Other residues
Analysing residues not classified as Protein/DNA/RNA/Water and splitting into groups...
Determining Verlet buffer for a tolerance of 0.005 kJ/mol/ps at 298 K
Calculated rlist for 1x1 atom pair-list as 1.200 nm, buffer size 0.000 nm
Set rlist, assuming 4x4 atom pair-list, to 1.200 nm, buffer size 0.000 nm
Note that mdrun will redetermine rlist based on the actual pair-list setup
Calculating fourier grid dimensions for X Y Z
Using a fourier grid of 24x24x25, spacing 0.110 0.110 0.117
This run will generate roughly 0 Mb of data



GROMACS reminds you: "A ship in port is safe, but that is not what ships are for. Sail out to sea and do new things." (Grace Hopper, developer of COBOL)

                      :-) GROMACS - gmx mdrun, 2019.3 (-:

                            GROMACS is written by:
     Emile Apol      Rossen Apostolov      Paul Bauer     Herman J.C. Berendsen
    Par Bjelkmar      Christian Blau   Viacheslav Bolnykh     Kevin Boyd    
 Aldert van Buuren   Rudi van Drunen     Anton Feenstra       Alan Gray     
  Gerrit Groenhof     Anca Hamuraru    Vincent Hindriksen  M. Eric Irrgang  
  Aleksei Iupinov   Christoph Junghans     Joe Jordan     Dimitrios Karkoulis
    Peter Kasson        Jiri Kraus      Carsten Kutzner      Per Larsson    
  Justin A. Lemkul    Viveca Lindahl    Magnus Lundborg     Erik Marklund   
    Pascal Merz     Pieter Meulenhoff    Teemu Murtola       Szilard Pall   
    Sander Pronk      Roland Schulz      Michael Shirts    Alexey Shvetsov  
   Alfons Sijbers     Peter Tieleman  

Analysing residue names:
There are:     1      Other residues
Analysing residues not classified as Protein/DNA/RNA/Water and splitting into groups...
Determining Verlet buffer for a tolerance of 0.005 kJ/mol/ps at 298 K
Calculated rlist for 1x1 atom pair-list as 1.200 nm, buffer size 0.000 nm
Set rlist, assuming 4x4 atom pair-list, to 1.200 nm, buffer size 0.000 nm
Note that mdrun will redetermine rlist based on the actual pair-list setup
Calculating fourier grid dimensions for X Y Z
Using a fourier grid of 24x24x25, spacing 0.110 0.110 0.117
This run will generate roughly 0 Mb of data


...

starting mdrun 'MOF'
0 steps,      0.0 ps.

Writing final coordinates.

NOTE: 19 % of the run time was spent in pair search,
      you might want to increase nstlist (this has no effect on accuracy)

               Core t (s)   Wall t (s)        (%)
       Time:       39.056        1.085     3599.2
                 (ns/day)    (hour/ns)
Performance:        0.080      301.425

GROMACS reminds you: "Let us not get carried away with our ideas and take our models too seriously" (Nancy Swanson)

                       :-) GROMACS - gmx dump, 2019.3 (-:

                            GROMACS is written by:
     Emile Apol      Rossen Apostolov      Paul Bauer     Herman J.C. Berendsen
    Par Bjelkmar      Christian Blau   Viacheslav Bolnykh     Kevin Boyd    
 Aldert van Buuren   Rudi van Drunen     Anton Feenstra       Alan Gray     
  Gerrit Groenhof     Anca Hamuraru    Vincent Hindriksen  M. Eric Irrgang  
  Aleksei Iupinov   Christoph Junghans     Joe Jordan     Dimitrios Karkoulis
    P

Analysing residue names:
There are:     1      Other residues
Analysing residues not classified as Protein/DNA/RNA/Water and splitting into groups...
Determining Verlet buffer for a tolerance of 0.005 kJ/mol/ps at 298 K
Calculated rlist for 1x1 atom pair-list as 1.200 nm, buffer size 0.000 nm
Set rlist, assuming 4x4 atom pair-list, to 1.200 nm, buffer size 0.000 nm
Note that mdrun will redetermine rlist based on the actual pair-list setup
Calculating fourier grid dimensions for X Y Z
Using a fourier grid of 24x24x25, spacing 0.110 0.110 0.117
This run will generate roughly 0 Mb of data


starting mdrun 'MOF'
0 steps,      0.0 ps.

Writing final coordinates.

NOTE: 27 % of the run time was spent in pair search,
      you might want to increase nstlist (this has no effect on accuracy)

               Core t (s)   Wall t (s)        (%)
       Time:       32.438        0.901     3599.1
                 (ns/day)    (hour/ns)
Performance:        0.096      250.356

GROMACS reminds you: "Christianity may be OK between consenting adults in private but should not be taught to young children." (Francis Crick)

                       :-) GROMACS - gmx dump, 2019.3 (-:

                            GROMACS is written by:
     Emile Apol      Rossen Apostolov      Paul Bauer     Herman J.C. Berendsen
    Par Bjelkmar      Christian Blau   Viacheslav Bolnykh     Kevin Boyd    
 Aldert van Buuren   Rudi van Drunen     Anton Feenstra       Alan Gray     
  Gerrit Groenhof     Anca Hamuraru    Vincent Hindriksen  M. Eric Irrgang  
  Aleksei Iupinov   Christoph Junghans     Joe Jordan   

Analysing residue names:
There are:     1      Other residues
Analysing residues not classified as Protein/DNA/RNA/Water and splitting into groups...
Determining Verlet buffer for a tolerance of 0.005 kJ/mol/ps at 298 K
Calculated rlist for 1x1 atom pair-list as 1.200 nm, buffer size 0.000 nm
Set rlist, assuming 4x4 atom pair-list, to 1.200 nm, buffer size 0.000 nm
Note that mdrun will redetermine rlist based on the actual pair-list setup
Calculating fourier grid dimensions for X Y Z
Using a fourier grid of 24x24x25, spacing 0.110 0.110 0.117
This run will generate roughly 0 Mb of data


starting mdrun 'MOF'
0 steps,      0.0 ps.

Writing final coordinates.

NOTE: 36 % of the run time was spent in pair search,
      you might want to increase nstlist (this has no effect on accuracy)

               Core t (s)   Wall t (s)        (%)
       Time:       39.630        1.101     3599.2
                 (ns/day)    (hour/ns)
Performance:        0.078      305.855

GROMACS reminds you: "Heavier-than-air flying machines are impossible." (Lord Kelvin, President of Royal Society, 1895.)

                       :-) GROMACS - gmx dump, 2019.3 (-:

                            GROMACS is written by:
     Emile Apol      Rossen Apostolov      Paul Bauer     Herman J.C. Berendsen
    Par Bjelkmar      Christian Blau   Viacheslav Bolnykh     Kevin Boyd    
 Aldert van Buuren   Rudi van Drunen     Anton Feenstra       Alan Gray     
  Gerrit Groenhof     Anca Hamuraru    Vincent Hindriksen  M. Eric Irrgang  
  Aleksei Iupinov   Christoph Junghans     Joe Jordan     Dimitrios Karkoulis
